# HT206B1: manual alignment and tumour-purity concordance
Prepared h5ad inputs must use spatial coordinates in µm. Supply the same cached feature cost and cell ordering as the experiment. `Tumor_purity_estimate` is an evaluation column, never an alignment input.
Repeat for U2–U3, U3–U4 and U4–U5; average section-pair scores equally.


## Data and local paths
Prepared revision inputs are not bundled with this notebook. Download sources and preparation instructions: [DATA.md](DATA.md).
Default prepared-input folder: `data/revision/ht206b1_u2_u3/` in the repository; set `RAFTUP_TUTORIAL_DATA` to override it. Relative paths inside `pair.json` resolve against that file. Outputs go to `outputs/revision/05_HT206B1_nonbrain/`; set `RAFTUP_OUTPUT_ROOT` to override. The real-data notebooks do not substitute synthetic data when inputs are missing.

## Input checklist and outputs

Follow [DATA.md](DATA.md) for HT206B1 access and preparation. Both slices need finite `Tumor_purity_estimate` values in `.obs`, matched by barcode. Coordinates must already be in µm (`pixels_per_100um=100`). The default case is U2–U3; other pairs require their own manifests, costs and output directories.

Run all cells in order. Save files are `OUT/HT206B1/<name>/alignment_matrix.npz` and `decision.json`. The last cell fits a rigid transform and prints purity Spearman, purity MAE and the number of spatial mutual-nearest-neighbour pairs within 100 µm. Metrics are displayed, not saved to a separate table. Averaging the three section-pair scores is a separate step; this notebook does not perform it automatically.

For terminal commands, kernel selection and environment-variable overrides, see [README.md](README.md). `OUT` below means the printed output root; the command-line runner supplies its own per-notebook output root.


In [ ]:
from pathlib import Path
import os, json
import numpy as np
import raftup
from raftup import run_automated_alignment, run_manual_alignment
from raftup.experiments import load_pair, save_alignment, coverage
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
DATA = Path(os.environ.get('RAFTUP_TUTORIAL_DATA', str(REPO / 'data/revision/ht206b1_u2_u3'))).expanduser().resolve()
OUT = Path(os.environ.get('RAFTUP_OUTPUT_ROOT', str(REPO / 'outputs/revision/05_HT206B1_nonbrain'))).expanduser().resolve()
print('Inputs:', DATA, '\nOutputs:', OUT)
print('RAFT-UP:', raftup.__version__, raftup.__file__)


In [ ]:
# Set DATA or RAFTUP_TUTORIAL_DATA to your prepared input directory.
# pair.json contains file paths, exact cost-matrix cell order and spatial calibration.
a, b, cost, cfg = load_pair(DATA / 'pair.json')
print(a.shape, b.shape, cost.shape)


In [ ]:
assert cfg['pixels_per_100um'] == 100, 'This example expects µm coordinates'
P, decision = run_manual_alignment(a, b, cost, technology='dlpfc', pixels_per_100um=100,
    downsampling_ratio=.067, feature_cutoff_downsample=.2,
    spatial_cutoff_downsample_um=400, feature_cutoff_recovery=.2,
    spatial_cutoff_recovery_um=200, k=20, random_state=42,
    min_downsampled_points=150, max_downsampled_points=800)
save_alignment(P, decision, OUT / 'HT206B1' / cfg['name'])


In [ ]:
from raftup.experiments import rigid_from_alignment, purity_concordance
R, t = rigid_from_alignment(P, a.obsm['spatial'], b.obsm['spatial'])
scores, i, j = purity_concordance(a.obsm['spatial'], b.obsm['spatial'] @ R + t,
    a.obs['Tumor_purity_estimate'], b.obs['Tumor_purity_estimate'], max_distance=100)
print(scores)


Use `figures/03_HT206B1_nonbrain.ipynb` for the archived histology masks, all-method purity comparison and top-five overlays. Competing methods require their own environments; their embeddings are not spatial coordinates.
